In [0]:
%sql
-- Utiliser le catalogue et le schema Gold
USE CATALOG `E-commerce`;
USE SCHEMA gold;

In [0]:
%sql
-- Afficher les tables Gold
SHOW TABLES;

###Total Revenue

In [0]:
%sql
-- Calculer le revenu total
SELECT
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM fact_sales;

#####Number of Transactions

In [0]:
%sql
-- Compter les transactions
SELECT
    COUNT(DISTINCT transaction_id) AS number_transactions
FROM fact_sales;

####Average Order Value

In [0]:
%sql
-- Calculer le panier moyen
SELECT
    ROUND(AVG(order_amount), 2) AS average_order_value
FROM (
    SELECT
        transaction_id,
        SUM(total_amount) AS order_amount
    FROM fact_sales
    GROUP BY transaction_id
);

#####Sales Trend

In [0]:
%sql
-- Calculer le revenu mensuel
SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS month,
    ROUND(SUM(total_amount), 2) AS revenue
FROM fact_sales
GROUP BY DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY month;

#####Objectif 2 — Analyze Product Performance

In [0]:
%sql
-- Calculer le revenu par produit
SELECT
    p.product_id,
    p.product_name,
    ROUND(SUM(s.total_amount), 2) AS revenue
FROM fact_sales s
LEFT JOIN dim_products p
    ON s.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY revenue DESC;

In [0]:
%sql
-- Trouver les produits les plus vendus
SELECT
    p.product_id,
    p.product_name,
    SUM(s.quantity) AS quantity_sold
FROM fact_sales s
LEFT JOIN dim_products p
    ON s.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY quantity_sold DESC;

In [0]:
%sql
-- Calculer le revenu par categorie
SELECT
    p.category,
    ROUND(SUM(s.total_amount), 2) AS revenue
FROM fact_sales s
LEFT JOIN dim_products p
    ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;

#####Objectif 3 — Understand Customer Behavior

In [0]:
%sql
-- Calculer le revenu par segment client
SELECT
    c.segment,
    ROUND(SUM(s.total_amount), 2) AS revenue
FROM fact_sales s
LEFT JOIN dim_customers c
    ON s.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY revenue DESC;

In [0]:
%sql
-- Calculer le revenu par pays
SELECT
    c.country,
    ROUND(SUM(s.total_amount), 2) AS revenue
FROM fact_sales s
LEFT JOIN dim_customers c
    ON s.customer_id = c.customer_id
GROUP BY c.country
ORDER BY revenue DESC;

In [0]:
%sql
-- Compter les sessions par client
SELECT
    customer_id,
    COUNT(DISTINCT session_id) AS number_sessions
FROM fact_sessions
GROUP BY customer_id
ORDER BY number_sessions DESC;

#####Objectif 4 — Measure Conversion Performance

In [0]:
%sql
-- Calculer le taux de conversion
SELECT
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM fact_sessions;

In [0]:
%sql
-- Calculer le taux de rebond
SELECT
    ROUND(AVG(bounced) * 100, 2) AS bounce_rate
FROM fact_sessions;

#####Objectif 5 — Analyze Customer Satisfaction

In [0]:
%sql
-- Calculer la note moyenne
SELECT
    ROUND(AVG(rating), 2) AS average_rating
FROM fact_reviews;

In [0]:
%sql
-- Trouver les produits les mieux notes
SELECT
    p.product_id,
    p.product_name,
    ROUND(AVG(r.rating), 2) AS average_rating,
    COUNT(r.review_id) AS number_reviews
FROM fact_reviews r
LEFT JOIN dim_products p
    ON r.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY average_rating DESC;

In [0]:
%sql
-- Trouver les produits les moins bien notes
SELECT
    p.product_id,
    p.product_name,
    ROUND(AVG(r.rating), 2) AS average_rating,
    COUNT(r.review_id) AS number_reviews
FROM fact_reviews r
LEFT JOIN dim_products p
    ON r.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY average_rating ASC;

###Objectif 6 — Analyze Marketing Channels

In [0]:
%sql
-- Analyser la performance des canaux
SELECT
    channel,
    COUNT(session_id) AS total_sessions,
    SUM(converted) AS total_conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM fact_sessions
GROUP BY channel
ORDER BY conversion_rate DESC;

#####Objectif 7 — Analyze Device Performance

In [0]:
%sql
-- Analyser la performance des appareils
SELECT
    device,
    COUNT(session_id) AS total_sessions,
    SUM(converted) AS total_conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate,
    ROUND(AVG(bounced) * 100, 2) AS bounce_rate
FROM fact_sessions
GROUP BY device
ORDER BY conversion_rate DESC;